# 視角切換與形狀偵測.py
**電腦視覺與深度學習 › Day14 OpenCV 電腦視覺**

這支程式把前幾支的處理步驟全部放在一起,還加上一排滑桿,可以即時切換原始、灰階、模糊、Canny、形狀偵測五種畫面,邊看邊調參數。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/7)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week1/DAY7"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '視角切換與形狀偵測.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    img = np.full((480, 640, 3), 255, np.uint8)
    cv2.circle(img, (130, 140), 70, (0, 0, 255), -1)
    cv2.circle(img, (520, 360), 55, (255, 0, 0), -1)
    cv2.rectangle(img, (260, 60), (400, 200), (0, 160, 0), -1)
    cv2.fillPoly(img, [np.array([[90, 420], [230, 420], [160, 300]])], (0, 200, 255))
    cv2.fillPoly(img, [np.array([[300, 300], [430, 280], [450, 420], [320, 440]])], (200, 0, 200))
    return img
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 說明與空的回呼函式
開頭說明這支程式能做什麼。nothing 是一個什麼都不做的函式,因為建立滑桿一定要給一個回呼函式,但我們每一圈自己去讀滑桿的值就好。

In [ ]:
# 即時 webcam 影像處理：可切換 原始/灰階/模糊/Canny/形狀偵測 五種畫面
# 所有參數用 slider (trackbar) 即時調整
# 形狀偵測模式下：可勾選要顯示哪些形狀（三角形、正方形/矩形、圓形、多邊形）

import cv2
import numpy as np

### 開鏡頭
打開 0 號鏡頭,並提醒你:按 q 離開,按 s 可以把目前畫面存成 snapshot.png。

In [ ]:
def nothing(x):
    """trackbar 的 callback，什麼都不做（我們每次 loop 自己讀值）"""
    pass


# ====== 開啟 webcam ======
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("錯誤：無法開啟視訊鏡頭")
    exit()

### 建立滑桿:視角、模糊、Canny
先開一個叫 Controls 的視窗,再用 createTrackbar 一個一個加滑桿。括號裡的兩個數字分別是預設值和最大值。模糊的 sigma 乘了 10,因為滑桿只能是整數,讀出來再除以 10。

In [ ]:
print("按 q 離開；按 s 存下目前畫面到 snapshot.png")


# ====== 建立控制面板 ======
cv2.namedWindow('Controls', cv2.WINDOW_NORMAL)
cv2.resizeWindow('Controls', 420, 520)

# 視角切換：0=原始, 1=灰階, 2=模糊, 3=Canny, 4=形狀偵測
cv2.createTrackbar('View 0RAW 1GRAY 2BLUR 3CANNY 4SHAPE', 'Controls', 4, 4, nothing)

# 模糊參數
cv2.createTrackbar('Blur ksize',       'Controls', 7,  30, nothing)   # 會強制轉奇數
cv2.createTrackbar('Blur sigma x10',   'Controls', 15, 100, nothing)  # 除以 10 → 0~10.0

# Canny 邊緣參數
cv2.createTrackbar('Canny T1',         'Controls', 30, 500, nothing)

### 面積、精度、圓度與開關
這裡加上面積範圍、多邊形逼近的精度、圓度門檻,還有四個形狀開關。圓度門檻 75 代表 0.75,圓度越接近 1 越像圓。

In [ ]:
cv2.createTrackbar('Canny T2',         'Controls', 150, 500, nothing)

# 形狀偵測面積範圍
cv2.createTrackbar('Min Area',         'Controls', 500,  20000, nothing)
cv2.createTrackbar('Max Area x10',     'Controls', 5000, 20000, nothing)  # ×10 → 上限 200000

# 多邊形逼近精準度 (approxPolyDP 的 epsilon 比例)
cv2.createTrackbar('Epsilon %',        'Controls', 4, 15, nothing)   # 除以 100 → 0.04

# 圓度門檻 (0.01 * value)：4πA / P² 越接近 1 越像圓
cv2.createTrackbar('Circle Thresh %',  'Controls', 75, 100, nothing)  # → 0.75

# 各形狀開關
cv2.createTrackbar('Show Triangle',    'Controls', 1, 1, nothing)
cv2.createTrackbar('Show Square/Rect', 'Controls', 1, 1, nothing)
cv2.createTrackbar('Show Circle',      'Controls', 1, 1, nothing)
cv2.createTrackbar('Show Polygon 5+',  'Controls', 0, 1, nothing)

### 主迴圈開始
VIEW_NAMES 是五種視角的名字,等一下寫在畫面左上角。主迴圈每圈先讀一格畫面。

In [ ]:
VIEW_NAMES = ["Raw (BGR)", "Gray", "Blur", "Canny", "Shape Detection"]

### 讀滑桿的值
每一圈都用 getTrackbarPos 讀出所有滑桿目前的位置。高斯模糊的大小一定要是奇數,所以如果讀到偶數就加 1。面積上限要乘 10、精度要除以 100,才換回真正的數值。

### 一次算好三個階段
每一圈都把灰階、模糊、Canny 三步全算好。這樣切換視角時不用重算,直接拿來顯示。

### 依視角挑要顯示的畫面
依照滑桿選的視角,決定顯示哪一張。灰階、模糊、邊緣圖都只有一個通道,所以要轉回三通道,才能在上面畫彩色的字。

### 形狀偵測:算圓度
形狀偵測模式會找輪廓、篩面積,再做凸包和多邊形逼近。新東西是圓度:4 乘 pi 乘面積,除以周長的平方。正圓是 1,正方形大約 0.785。

### 分類規則與寬高比
判斷順序是圓、三角形、四邊形、多邊形。圓要同時滿足圓度夠高、而且頂點至少 5 個,這樣正方形就不會被誤認成圓。四邊形再用寬高比分:0.85 到 1.15 之間是正方形,其他是長方形。

### 畫出形狀並統計
有分類到才畫外框、中心點和名稱。最後在右上角列出每種形狀各有幾個。

### 標題、存圖與離開
左上角寫目前視角的名字,下面寫操作提示。按 s 用 imwrite 存圖,按 q 離開,最後放掉鏡頭。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        print("錯誤：無法讀取畫面")
        break

    # ====== 讀 trackbar ======
    view       = cv2.getTrackbarPos('View 0RAW 1GRAY 2BLUR 3CANNY 4SHAPE', 'Controls')
    ksize      = max(1, cv2.getTrackbarPos('Blur ksize', 'Controls'))
    if ksize % 2 == 0:
        ksize += 1                             # Gaussian ksize 必須是奇數
    sigma      = cv2.getTrackbarPos('Blur sigma x10', 'Controls') / 10.0
    t1         = cv2.getTrackbarPos('Canny T1', 'Controls')
    t2         = cv2.getTrackbarPos('Canny T2', 'Controls')
    min_area   = cv2.getTrackbarPos('Min Area', 'Controls')
    max_area   = cv2.getTrackbarPos('Max Area x10', 'Controls') * 10
    eps_pct    = max(cv2.getTrackbarPos('Epsilon %', 'Controls'), 1) / 100.0
    circ_thr   = cv2.getTrackbarPos('Circle Thresh %', 'Controls') / 100.0
    show_tri   = cv2.getTrackbarPos('Show Triangle',    'Controls')
    show_sq    = cv2.getTrackbarPos('Show Square/Rect', 'Controls')
    show_ci    = cv2.getTrackbarPos('Show Circle',      'Controls')
    show_pg    = cv2.getTrackbarPos('Show Polygon 5+',  'Controls')

    # ====== 影像處理 pipeline ======
    # 每一階段都算，方便切換不用重跑
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (ksize, ksize), sigma)
    edges = cv2.Canny(blur, t1, t2)

    # ====== 依 view 選擇要顯示的畫面 ======
    if view == 0:
        display = frame.copy()
    elif view == 1:
        display = cv2.cvtColor(gray,  cv2.COLOR_GRAY2BGR)
    elif view == 2:
        display = cv2.cvtColor(blur,  cv2.COLOR_GRAY2BGR)
    elif view == 3:
        display = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    else:
        # ====== Shape Detection 模式 ======
        display = frame.copy()
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        counts = {"Triangle": 0, "Square": 0, "Rectangle": 0, "Circle": 0, "Polygon": 0}

        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < min_area or area > max_area:
                continue

            perimeter = cv2.arcLength(cnt, True)
            if perimeter <= 0:
                continue

            # 凸包 + 多邊形逼近
            hull   = cv2.convexHull(cnt)
            eps    = eps_pct * cv2.arcLength(hull, True)
            approx = cv2.approxPolyDP(hull, eps, True)
            vtx    = len(approx)

            # 圓度：4πA / P²，圓形時接近 1
            circularity = 4 * np.pi * area / (perimeter * perimeter)

            # 中心點
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])

            shape = None
            color = (255, 255, 255)

            # 判斷順序：圓 → 三角形 → 四邊形 → 多邊形
            if show_ci and circularity > circ_thr and vtx >= 5:
                shape = "Circle"
                color = (0, 0, 255)                  # 紅
                counts["Circle"] += 1
            elif show_tri and vtx == 3:
                shape = "Triangle"
                color = (0, 255, 255)                # 黃
                counts["Triangle"] += 1
            elif show_sq and vtx == 4:
                x, y, w, h = cv2.boundingRect(approx)
                ratio = w / float(h) if h > 0 else 0
                if 0.85 <= ratio <= 1.15:
                    shape = "Square"
                    counts["Square"] += 1
                else:
                    shape = "Rectangle"
                    counts["Rectangle"] += 1
                color = (0, 255, 0)                  # 綠
            elif show_pg and vtx >= 5:
                shape = f"Polygon-{vtx}"
                color = (255, 100, 200)              # 粉
                counts["Polygon"] += 1

            if shape:
                cv2.drawContours(display, [approx], -1, color, 2)
                cv2.circle(display, (cX, cY), 4, color, -1)
                cv2.putText(display, f"{shape}  A:{int(area)}",
                            (cX - 55, cY - 12),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # 右上角統計數量
        stat_y = 30
        for name, n in counts.items():
            if n > 0:
                cv2.putText(display, f"{name}: {n}",
                            (display.shape[1] - 180, stat_y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                stat_y += 22

    # ====== 頂部 label + 操作提示 ======
    cv2.putText(display, VIEW_NAMES[view], (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(display, "q=Quit  s=Save snapshot.png",
                (10, display.shape[0] - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imshow('Preview', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        cv2.imwrite('snapshot.png', display)
        print("已存 snapshot.png")

cap.release()
cv2.destroyAllWindows()

## 小結
你學會用滑桿即時調參數,也看到用頂點數、圓度、寬高比三種線索分類。下一支會把同樣的功能包進 tkinter 視窗,做成真正的工具。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/7) 看逐句拆解、做練習。